# Phase 1 self-labeling live train\n\nRun the cells from top to bottom. The training cell updates loss and best-source plots after every episode.

In [ ]:
from pathlib import Path\nimport sys\n\nROOT = Path.cwd()\nif not (ROOT / 'main.py').exists():\n    ROOT = ROOT.parent\nsys.path.insert(0, str(ROOT))\n\nfrom IPython.display import clear_output, display\nimport matplotlib.pyplot as plt\n\nfrom main import build_environment, _phase1_bay_ids_from_env\nfrom Train.algorithm.phase1_self_labeling import PHASE1_SELF_LABEL_HEURISTIC_BANK, train_phase1_pointer_self_labeling\nfrom Utils.phase1_block_data_generator import load_phase1_actual_blocks\nfrom Utils.phase1_episode_dataset import build_phase1_episode_jobs\n

In [ ]:
CONFIG = 'config_np_100.yaml'\nBAY_IDS = '22,23,24'\nBLOCK_XLSX = 'input/절단03~04_NP물량_마스킹_블록_수정_260618.xlsx'\nGYEL = 'NP'\nOUTPUT_DIR = 'output/phase1_self_labeling_sampled_notebook'\nEPISODES = 20\nMIN_BLOCKS = 12\nMAX_BLOCKS = 80\nNOISE_RATIO = 0.03\nROLLOUT_SAMPLES = 4\nHIDDEN_DIM = 64\nLR = 0.001\nTEMPERATURE = 1.0\nSEED = 0\nHEURISTICS = PHASE1_SELF_LABEL_HEURISTIC_BANK\n

In [ ]:
def live_plot(metrics_rows, candidate_rows):\n    clear_output(wait=True)\n    episodes = [int(row['episode']) for row in metrics_rows]\n    losses = [float(row['loss']) for row in metrics_rows]\n    best_rows = [row for row in candidate_rows if int(row.get('is_best', 0)) == 1]\n    score0 = [float(row['score_0']) for row in best_rows]\n    source_counts = {}\n    for row in metrics_rows:\n        source_counts[row['best_source']] = source_counts.get(row['best_source'], 0) + 1\n\n    fig, axes = plt.subplots(1, 3, figsize=(16, 4))\n    axes[0].plot(episodes, losses, marker='o')\n    axes[0].set_title('Loss')\n    axes[0].set_xlabel('Episode')\n    axes[0].grid(True, alpha=0.3)\n\n    axes[1].plot(episodes, score0, marker='o')\n    axes[1].set_title('Best primary score')\n    axes[1].set_xlabel('Episode')\n    axes[1].grid(True, alpha=0.3)\n\n    axes[2].bar(list(source_counts), list(source_counts.values()))\n    axes[2].set_title('Best source count')\n    axes[2].tick_params(axis='x', rotation=20)\n\n    plt.tight_layout()\n    display(fig)\n    plt.close(fig)\n    print(f'episode={episodes[-1]} loss={losses[-1]:.6f} best_source={metrics_rows[-1]["best_source"]}')\n

In [ ]:
env = build_environment(CONFIG)\nbay_ids = _phase1_bay_ids_from_env(env, BAY_IDS)\nactual_blocks = load_phase1_actual_blocks(BLOCK_XLSX, gyel=GYEL)\nepisode_specs = build_phase1_episode_jobs(\n    actual_blocks=actual_blocks,\n    episode_count=EPISODES,\n    min_blocks=MIN_BLOCKS,\n    max_blocks=MAX_BLOCKS,\n    seed=SEED,\n    noise_ratio=NOISE_RATIO,\n)\nepisode_jobs = [spec['jobs'] for spec in episode_specs]\nepisode_metadata = [\n    {'episode_id': spec['episode_id'], 'problem_id': spec['problem_id'], 'block_count': spec['block_count'], 'seed': spec['seed']}\n    for spec in episode_specs\n]\nsummary = train_phase1_pointer_self_labeling(\n    jobs=None,\n    episode_jobs=episode_jobs,\n    episode_metadata=episode_metadata,\n    bay_ids=bay_ids,\n    output_dir=OUTPUT_DIR,\n    episodes=EPISODES,\n    rollout_samples=ROLLOUT_SAMPLES,\n    heuristic_algorithms=HEURISTICS,\n    lr=LR,\n    hidden_dim=HIDDEN_DIM,\n    temperature=TEMPERATURE,\n    seed=SEED,\n    metrics_callback=live_plot,\n)\nsummary\n